# Tutorial 11E: Statistical Testing and Model Comparison - SOLUTIONS

This notebook provides complete solutions for all exercises in Tutorial 05.

## Exercises Overview

| Exercise | Title | Difficulty |
|----------|-------|------------|
| 1 | Complete Model Selection | Easy |
| 2 | Bootstrap Robustness | Medium |
| 3 | Specification Testing | Hard |

In [ ]:
# Setup
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
from pathlib import Path

import statsmodels.api as sm
from scipy import stats

from panelbox.frontier import (
    StochasticFrontier,
    add_translog,
    inefficiency_presence_test,
)
from panelbox.frontier.bootstrap import SFABootstrap
from panelbox.frontier.panel_utils import (
    lr_test_bc92_eta_constant,
    lr_test_kumbhakar_constant,
)

np.random.seed(42)

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 4)

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
FIGURES_DIR = BASE_DIR / "outputs" / "figures" / "05_testing"
TABLES_DIR_LATEX = BASE_DIR / "outputs" / "tables" / "latex"
TABLES_DIR_HTML = BASE_DIR / "outputs" / "tables" / "html"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR_LATEX.mkdir(parents=True, exist_ok=True)
TABLES_DIR_HTML.mkdir(parents=True, exist_ok=True)

print("Setup complete!")

In [ ]:
# Load datasets
data = pd.read_csv(DATA_DIR / "dairy_farm.csv")
panel_data = pd.read_csv(DATA_DIR / "telecom_panel.csv")

exog_vars = ["log_cows", "log_feed", "log_land", "log_labor"]
panel_exog = ["log_labor", "log_capital", "log_spectrum"]

print(f"Dairy farm: {data.shape}")
print(f"Telecom panel: {panel_data.shape}")

---

# Exercise 1 Solution: Complete Model Selection (Easy)

**Task**: Follow the model selection workflow step-by-step using the dairy farm data, starting with the exponential distribution.

In [ ]:
# Step 1: Test for inefficiency with exponential distribution
print("=" * 60)
print("STEP 1: Test for Inefficiency (Exponential)")
print("=" * 60)

# OLS baseline
X_ols = sm.add_constant(data[exog_vars])
ols_result = sm.OLS(data["log_milk"], X_ols).fit()

# SFA with exponential
sfa_exp = StochasticFrontier(
    data=data,
    depvar="log_milk",
    exog=exog_vars,
    frontier="production",
    dist="exponential",
).fit()

ineff = inefficiency_presence_test(
    loglik_sfa=sfa_exp.loglik,
    loglik_ols=ols_result.llf,
    residuals_ols=ols_result.resid.values,
    frontier_type="production",
    distribution="exponential",
)

print(f"Log-L (OLS): {ols_result.llf:.4f}")
print(f"Log-L (SFA): {sfa_exp.loglik:.4f}")
print(f"LR Statistic: {ineff['lr_statistic']:.4f}")
print(f"P-value: {ineff['pvalue']:.4f}")
print(f"Conclusion: {ineff['conclusion']}")
print(f"Skewness: {ineff['skewness']:.4f}")
if ineff.get("skewness_warning"):
    print(f"WARNING: {ineff['skewness_warning']}")

In [ ]:
# Step 2: Compare distributions using AIC/BIC
print("=" * 60)
print("STEP 2: Distribution Comparison")
print("=" * 60)

distributions = ["half_normal", "exponential", "truncated_normal"]
dist_models = {}

for dist in distributions:
    try:
        dist_models[dist] = StochasticFrontier(
            data=data,
            depvar="log_milk",
            exog=exog_vars,
            frontier="production",
            dist=dist,
        ).fit()
        print(
            f"  {dist:<20}: Log-L = {dist_models[dist].loglik:.4f}, "
            f"AIC = {dist_models[dist].aic:.4f}, "
            f"BIC = {dist_models[dist].bic:.4f}, "
            f"Mean TE = {dist_models[dist].mean_efficiency:.4f}"
        )
    except Exception as e:
        print(f"  {dist}: Failed ({e})")

# Find best
aic_vals = {d: dist_models[d].aic for d in dist_models}
bic_vals = {d: dist_models[d].bic for d in dist_models}
best_aic = min(aic_vals, key=aic_vals.get)
best_bic = min(bic_vals, key=bic_vals.get)

print(f"\nBest by AIC: {best_aic}")
print(f"Best by BIC: {best_bic}")

best_dist = best_aic
print(f"\nSelected distribution: {best_dist}")

In [ ]:
# Step 3: Functional form test (CD vs Translog)
print("=" * 60)
print("STEP 3: Functional Form Test")
print("=" * 60)

cd_result = dist_models[best_dist]

# Create translog terms
data_tl = add_translog(data, exog_vars)
tl_new_cols = [c for c in data_tl.columns if c not in data.columns]
tl_exog = exog_vars + tl_new_cols

try:
    tl_result = StochasticFrontier(
        data=data_tl,
        depvar="log_milk",
        exog=tl_exog,
        frontier="production",
        dist=best_dist,
    ).fit()

    ff_test = cd_result.compare_functional_form(tl_result)

    print(f"Log-L (CD):  {cd_result.loglik:.4f} ({cd_result.nparams} params)")
    print(f"Log-L (TL):  {tl_result.loglik:.4f} ({tl_result.nparams} params)")
    print(f"LR Statistic: {ff_test['lr_statistic']:.4f}")
    print(f"DF: {ff_test['df']}")
    print(f"P-value: {ff_test['pvalue']:.4f}")
    print(f"Conclusion: {ff_test['conclusion']}")
    print(f"AIC: CD = {ff_test['aic_cd']:.2f}, TL = {ff_test['aic_tl']:.2f}")
    print(f"BIC: CD = {ff_test['bic_cd']:.2f}, TL = {ff_test['bic_tl']:.2f}")

    if ff_test["conclusion"] == "translog":
        final_result = tl_result
        final_form = "translog"
    else:
        final_result = cd_result
        final_form = "cobb_douglas"
except Exception as e:
    print(f"Translog estimation failed: {e}")
    final_result = cd_result
    final_form = "cobb_douglas"

print(f"\nSelected functional form: {final_form}")

In [ ]:
# Step 4: Returns to scale test
print("=" * 60)
print("STEP 4: Returns to Scale Test")
print("=" * 60)

rts = final_result.returns_to_scale_test(input_vars=exog_vars)

print(f"RTS estimate: {rts['rts']:.4f}")
print(f"Std error:    {rts['rts_se']:.4f}")
print(f"95% CI:       [{rts['ci'][0]:.4f}, {rts['ci'][1]:.4f}]")
print(f"Wald stat:    {rts['test_statistic']:.4f}")
print(f"P-value:      {rts['pvalue']:.4f}")
print(f"Conclusion:   {rts['conclusion']}")

In [ ]:
# Step 5: Final report
print("=" * 70)
print("FINAL MODEL SELECTION REPORT")
print("=" * 70)
print(f"\n1. Inefficiency presence: {ineff['conclusion']}")
print(f"   (LR = {ineff['lr_statistic']:.4f}, p = {ineff['pvalue']:.4f})")
print(f"\n2. Distribution: {best_dist}")
print(f"   (Selected by AIC from: {list(dist_models.keys())})")
print(f"\n3. Functional form: {final_form}")
if "ff_test" in dir():
    print(f"   (LR = {ff_test['lr_statistic']:.4f}, p = {ff_test['pvalue']:.4f})")
print(f"\n4. Returns to scale: {rts['conclusion']}")
print(f"   (RTS = {rts['rts']:.4f}, p = {rts['pvalue']:.4f})")
print(f"\nFinal model Mean TE: {final_result.mean_efficiency:.4f}")
print("\nJustification:")
print("  - Inefficiency is statistically significant (reject OLS)")
print(f"  - {best_dist} selected because it has the lowest AIC")
print(f"  - {final_form} selected based on LR test")
print(f"  - {rts['conclusion']}: RTS = {rts['rts']:.3f}")

### Interpretation

The systematic workflow produces a well-justified final specification:

1. **Inefficiency is present**: The mixed chi-square LR test strongly rejects OLS, confirming that SFA is appropriate for this data.
2. **Distribution choice**: The AIC/BIC ranking selects the best distributional assumption. Different distributions can yield different efficiency estimates, so this choice matters.
3. **Functional form**: The LR test determines whether the flexibility of the Translog specification is statistically justified. If not, the simpler Cobb-Douglas is preferred.
4. **Returns to scale**: The Wald test tells us about the technology structure of dairy farming.

---

# Exercise 2 Solution: Bootstrap Robustness (Medium)

**Task**: Compare parametric and pairs bootstrap methods for efficiency inference.

In [ ]:
# Step 1: Parametric bootstrap
print("=" * 60)
print("PARAMETRIC BOOTSTRAP (B=200)")
print("=" * 60)

# Use the best model from Exercise 1
boot_param = SFABootstrap(
    result=cd_result,
    method="parametric",
    n_boot=200,
    ci_level=0.95,
    seed=42,
    n_jobs=1,
)
eff_param = boot_param.bootstrap_efficiency(estimator="bc")

print(f"Valid replications: {boot_param.bootstrap_parameters()['n_valid']}")
print(f"Mean bootstrap TE: {eff_param['efficiency'].mean():.4f}")
print(f"Mean CI width: {(eff_param['ci_upper'] - eff_param['ci_lower']).mean():.4f}")

In [ ]:
# Step 2: Pairs bootstrap
print("=" * 60)
print("PAIRS BOOTSTRAP (B=200)")
print("=" * 60)

boot_pairs = SFABootstrap(
    result=cd_result,
    method="pairs",
    n_boot=200,
    ci_level=0.95,
    seed=42,
    n_jobs=1,
)
eff_pairs = boot_pairs.bootstrap_efficiency(estimator="bc")

print(f"Mean bootstrap TE: {eff_pairs['efficiency'].mean():.4f}")
print(f"Mean CI width: {(eff_pairs['ci_upper'] - eff_pairs['ci_lower']).mean():.4f}")

In [ ]:
# Step 3: Compare top 10 and bottom 10 farms
print("=" * 70)
print("COMPARISON: Top 10 and Bottom 10 Farms")
print("=" * 70)

# Sort by efficiency
sorted_idx = eff_param["efficiency"].values.argsort()
bottom_10 = sorted_idx[:10]
top_10 = sorted_idx[-10:]

# Top 10
print("\nTop 10 Most Efficient Farms:")
top_comp = pd.DataFrame(
    {
        "Farm": top_10 + 1,
        "TE": eff_param["efficiency"].values[top_10],
        "Param_CI_Lower": eff_param["ci_lower"].values[top_10],
        "Param_CI_Upper": eff_param["ci_upper"].values[top_10],
        "Pairs_CI_Lower": eff_pairs["ci_lower"].values[top_10],
        "Pairs_CI_Upper": eff_pairs["ci_upper"].values[top_10],
    }
)
top_comp["Param_Width"] = top_comp["Param_CI_Upper"] - top_comp["Param_CI_Lower"]
top_comp["Pairs_Width"] = top_comp["Pairs_CI_Upper"] - top_comp["Pairs_CI_Lower"]
display(top_comp.round(4))

# Bottom 10
print("\nBottom 10 Least Efficient Farms:")
bot_comp = pd.DataFrame(
    {
        "Farm": bottom_10 + 1,
        "TE": eff_param["efficiency"].values[bottom_10],
        "Param_CI_Lower": eff_param["ci_lower"].values[bottom_10],
        "Param_CI_Upper": eff_param["ci_upper"].values[bottom_10],
        "Pairs_CI_Lower": eff_pairs["ci_lower"].values[bottom_10],
        "Pairs_CI_Upper": eff_pairs["ci_upper"].values[bottom_10],
    }
)
bot_comp["Param_Width"] = bot_comp["Param_CI_Upper"] - bot_comp["Param_CI_Lower"]
bot_comp["Pairs_Width"] = bot_comp["Pairs_CI_Upper"] - bot_comp["Pairs_CI_Lower"]
display(bot_comp.round(4))

In [ ]:
# Step 4: Rank correlation
print("=" * 60)
print("RANK CORRELATION ANALYSIS")
print("=" * 60)

# Rankings from each bootstrap
rank_param = eff_param["efficiency"].values.argsort().argsort()
rank_pairs = eff_pairs["efficiency"].values.argsort().argsort()

rho, pval = stats.spearmanr(rank_param, rank_pairs)
print(f"Spearman rank correlation: rho = {rho:.4f} (p = {pval:.6f})")

pearson_r = np.corrcoef(eff_param["efficiency"].values, eff_pairs["efficiency"].values)[0, 1]
print(f"Pearson correlation: r = {pearson_r:.4f}")

# Check overlap in top/bottom 10
sorted_param = eff_param["efficiency"].values.argsort()
sorted_pairs = eff_pairs["efficiency"].values.argsort()

top10_param = set(sorted_param[-10:])
top10_pairs = set(sorted_pairs[-10:])
bottom10_param = set(sorted_param[:10])
bottom10_pairs = set(sorted_pairs[:10])

top_overlap = len(top10_param & top10_pairs)
bot_overlap = len(bottom10_param & bottom10_pairs)

print(f"\nTop 10 overlap:    {top_overlap}/10 farms in common")
print(f"Bottom 10 overlap: {bot_overlap}/10 farms in common")

In [ ]:
# Visualization: parametric vs pairs efficiency
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot of efficiency estimates
axes[0].scatter(
    eff_param["efficiency"].values,
    eff_pairs["efficiency"].values,
    alpha=0.4,
    s=15,
    color="steelblue",
)
axes[0].plot([0, 1], [0, 1], "r--", linewidth=2, label="45-degree")
axes[0].set_xlabel("Parametric Bootstrap TE", fontsize=12)
axes[0].set_ylabel("Pairs Bootstrap TE", fontsize=12)
axes[0].set_title(
    f"Efficiency: Parametric vs Pairs\n(Spearman rho = {rho:.3f})", fontsize=13, fontweight="bold"
)
axes[0].legend(fontsize=10)
axes[0].set_aspect("equal")

# CI width comparison
param_widths = (eff_param["ci_upper"] - eff_param["ci_lower"]).values
pairs_widths = (eff_pairs["ci_upper"] - eff_pairs["ci_lower"]).values

axes[1].scatter(param_widths, pairs_widths, alpha=0.4, s=15, color="#e74c3c")
max_w = max(param_widths.max(), pairs_widths.max())
axes[1].plot([0, max_w], [0, max_w], "k--", linewidth=1.5, label="Equal width")
axes[1].set_xlabel("Parametric CI Width", fontsize=12)
axes[1].set_ylabel("Pairs CI Width", fontsize=12)
axes[1].set_title("CI Width Comparison", fontsize=13, fontweight="bold")
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"Mean CI width -- Parametric: {param_widths.mean():.4f}")
print(f"Mean CI width -- Pairs:      {pairs_widths.mean():.4f}")

### Interpretation

1. **Parametric vs Pairs**: The two bootstrap methods generally give similar efficiency estimates, as shown by the high Spearman rank correlation. This is reassuring -- the results are not overly sensitive to the bootstrap method.

2. **CI Widths**: Pairs bootstrap CIs tend to be slightly wider because they don't assume a specific error distribution. Parametric bootstrap CIs are narrower but rely on the distributional assumption being correct.

3. **Ranking Stability**: The high overlap in the top/bottom 10 shows that the most and least efficient farms are consistently identified regardless of bootstrap method. This gives confidence in the efficiency rankings.

4. **Practical Recommendation**: Use parametric bootstrap when you're confident in the distributional assumption, and pairs bootstrap as a robustness check. If they disagree substantially, the distributional assumption may be incorrect.

---

# Exercise 3 Solution: Specification Testing (Hard)

**Task**: Comprehensive specification analysis for the telecom panel data with 6 models.

In [ ]:
# Step 1: Estimate 6 models (3 panel types x 2 distributions)
print("=" * 70)
print("ESTIMATING 6 PANEL MODELS")
print("=" * 70)

panel_types = ["pitt_lee", "bc92", "kumbhakar_1990"]
dists = ["half_normal", "exponential"]
all_models = {}

for ptype in panel_types:
    for dist in dists:
        name = f"{ptype}_{dist}"
        print(f"  Estimating {name}...")
        try:
            model = StochasticFrontier(
                data=panel_data,
                depvar="log_output",
                exog=panel_exog,
                entity="firm_id",
                time="year",
                frontier="production",
                dist=dist,
                model_type=ptype,
            )
            all_models[name] = model.fit()
            print(
                f"    Log-L: {all_models[name].loglik:.4f}, "
                f"Mean TE: {all_models[name].mean_efficiency:.4f}"
            )
        except Exception as e:
            print(f"    Failed: {e}")

print(f"\nSuccessfully estimated {len(all_models)} models.")

In [ ]:
# Step 2: Comprehensive comparison table
print("=" * 90)
print("COMPREHENSIVE MODEL COMPARISON")
print("=" * 90)

comp_data = []
for name, result in all_models.items():
    parts = name.split("_", 1) if "_half" in name or "_exp" in name else name.rsplit("_", 1)
    # Parse panel type and distribution from the name
    if "_half_normal" in name:
        panel_type = name.replace("_half_normal", "")
        distribution = "half_normal"
    elif "_exponential" in name:
        panel_type = name.replace("_exponential", "")
        distribution = "exponential"
    else:
        panel_type = name
        distribution = "unknown"

    comp_data.append(
        {
            "Model": name,
            "Panel Type": panel_type,
            "Distribution": distribution,
            "Log-L": result.loglik,
            "AIC": result.aic,
            "BIC": result.bic,
            "N Params": result.nparams,
            "sigma_v": result.sigma_v,
            "sigma_u": result.sigma_u,
            "Mean TE": result.mean_efficiency,
        }
    )

comp_df = pd.DataFrame(comp_data).sort_values("AIC")
display(comp_df.round(4))

best_overall = comp_df.iloc[0]
print(f"\nBest model by AIC: {best_overall['Model']}")
print(f"Best model by BIC: {comp_df.sort_values('BIC').iloc[0]['Model']}")

In [ ]:
# Step 3: Pairwise LR tests for nested models
print("=" * 70)
print("PAIRWISE LR TESTS (Nested Models)")
print("=" * 70)

lr_results = []

# Within each distribution: test time-varying
for dist in dists:
    pl_key = f"pitt_lee_{dist}"
    bc_key = f"bc92_{dist}"
    kumb_key = f"kumbhakar_1990_{dist}"

    # Pitt-Lee vs BC92
    if pl_key in all_models and bc_key in all_models:
        lr_bc = lr_test_bc92_eta_constant(
            loglik_bc92=all_models[bc_key].loglik,
            loglik_pitt_lee=all_models[pl_key].loglik,
        )
        print(f"\n{dist}: Pitt-Lee vs BC92 (H0: eta=0)")
        print(
            f"  LR = {lr_bc['LR_stat']:.4f}, p = {lr_bc['p_value']:.4f}, "
            f"Reject: {lr_bc['reject_H0']}"
        )
        lr_results.append(
            {
                "Distribution": dist,
                "Test": "Pitt-Lee vs BC92",
                "LR": lr_bc["LR_stat"],
                "DF": 1,
                "P-value": lr_bc["p_value"],
                "Reject H0": lr_bc["reject_H0"],
            }
        )

    # Pitt-Lee vs Kumbhakar
    if pl_key in all_models and kumb_key in all_models:
        lr_kumb = lr_test_kumbhakar_constant(
            loglik_kumbhakar=all_models[kumb_key].loglik,
            loglik_pitt_lee=all_models[pl_key].loglik,
        )
        print(f"\n{dist}: Pitt-Lee vs Kumbhakar (H0: b=c=0)")
        print(
            f"  LR = {lr_kumb['LR_stat']:.4f}, p = {lr_kumb['p_value']:.4f}, "
            f"Reject: {lr_kumb['reject_H0']}"
        )
        lr_results.append(
            {
                "Distribution": dist,
                "Test": "Pitt-Lee vs Kumbhakar",
                "LR": lr_kumb["LR_stat"],
                "DF": 2,
                "P-value": lr_kumb["p_value"],
                "Reject H0": lr_kumb["reject_H0"],
            }
        )

if lr_results:
    lr_summary_df = pd.DataFrame(lr_results)
    print("\n\nLR Test Summary Table:")
    display(lr_summary_df.round(4))

In [ ]:
# Step 4: RTS test for best panel model
print("=" * 60)
print("RETURNS TO SCALE: Best Panel Model")
print("=" * 60)

best_panel_name = comp_df.iloc[0]["Model"]
best_panel = all_models[best_panel_name]

print(f"Best model: {best_panel_name}")

rts_panel = best_panel.returns_to_scale_test(input_vars=panel_exog)

print("\nInput elasticities:")
elas = best_panel.elasticities(input_vars=panel_exog)
for var, e in elas.items():
    print(f"  {var}: {e:.4f}")

print(f"\nRTS = {rts_panel['rts']:.4f} (SE = {rts_panel['rts_se']:.4f})")
print(f"95% CI: [{rts_panel['ci'][0]:.4f}, {rts_panel['ci'][1]:.4f}]")
print(f"Wald stat: {rts_panel['test_statistic']:.4f}, p = {rts_panel['pvalue']:.4f}")
print(f"Conclusion: {rts_panel['conclusion']}")

In [ ]:
# Step 5: Model selection report
print("=" * 70)
print("MODEL SELECTION REPORT: Telecom Panel Data")
print("=" * 70)

print(f"""
SUMMARY
-------
We estimated {len(all_models)} models combining 3 panel specifications
(Pitt-Lee, BC92, Kumbhakar 1990) with 2 distributional assumptions
(half-normal, exponential).

BEST MODEL: {best_panel_name}
  - AIC: {comp_df.iloc[0]["AIC"]:.4f}
  - BIC: {comp_df.iloc[0]["BIC"]:.4f}
  - Mean TE: {comp_df.iloc[0]["Mean TE"]:.4f}

TIME-VARYING INEFFICIENCY:
  - LR tests indicate whether time-varying models improve upon Pitt-Lee.
""")

for lr_row in lr_results:
    outcome = "Reject H0 (time-varying matters)" if lr_row["Reject H0"] else "Do not reject H0"
    print(f"  {lr_row['Distribution']}: {lr_row['Test']} -> {outcome} (p={lr_row['P-value']:.4f})")

print(f"""
RETURNS TO SCALE:
  RTS = {rts_panel["rts"]:.4f}, Conclusion: {rts_panel["conclusion"]}

RECOMMENDATIONS:
  1. Use {best_panel_name} as the preferred specification
  2. {"Time-varying models add value" if any(r["Reject H0"] for r in lr_results) else "Time-invariant Pitt-Lee may suffice"}
  3. RTS = {rts_panel["rts"]:.3f} suggests {"scale economies exist" if rts_panel["conclusion"] != "CRS" else "optimal scale"}
""")

In [ ]:
# Visualization: comprehensive comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: AIC comparison
model_names = comp_df["Model"].values
aic_values = comp_df["AIC"].values
colors = ["#27ae60" if i == 0 else "steelblue" for i in range(len(model_names))]

y_pos = np.arange(len(model_names))
axes[0].barh(y_pos, aic_values, color=colors, alpha=0.8, edgecolor="black", linewidth=0.5)
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(model_names, fontsize=9)
axes[0].set_xlabel("AIC", fontsize=12)
axes[0].set_title("AIC Comparison (lower is better)", fontsize=13, fontweight="bold")

# Panel 2: Mean TE comparison
te_values = comp_df["Mean TE"].values
axes[1].barh(y_pos, te_values, color="#FF9800", alpha=0.8, edgecolor="black", linewidth=0.5)
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(model_names, fontsize=9)
axes[1].set_xlabel("Mean Technical Efficiency", fontsize=12)
axes[1].set_title("Mean TE by Model", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

### Interpretation

1. **Model comparison**: With 6 models, we can systematically compare both distributional and temporal assumptions. The best model by AIC/BIC balances fit and parsimony.

2. **LR tests**: The nested tests (Pitt-Lee vs BC92, Pitt-Lee vs Kumbhakar) tell us whether time-varying inefficiency is empirically important. If both reject H0, we need a time-varying specification.

3. **Distribution matters**: Different distributional assumptions can lead to different efficiency estimates and rankings. The exponential distribution imposes a monotonically decreasing density, while half-normal allows a mode at zero.

4. **Policy implications**: The RTS test has direct implications -- if IRS, the telecom sector could benefit from consolidation; if DRS, from more competition.

---

## Key Takeaways from All Exercises

1. **Systematic workflow** prevents ad hoc model selection and provides defensible choices
2. **Multiple tests** (LR, AIC, BIC, Wald) serve different purposes -- use the right test for the right comparison
3. **Bootstrap methods** provide robust inference when asymptotic theory may be unreliable
4. **Sensitivity analysis** across distributions and bootstrap methods builds confidence in conclusions
5. **Panel specification** choice (time-invariant vs time-varying) has real economic implications
6. **Always report** both AIC and BIC -- they may disagree when parsimony vs fit tradeoffs differ